# 04 - Train PSA + ECA ResNet

This notebook trains the custom ResNet-style PSA + ECA model.

Current plan:
1. Load leakage-safe train/val/test split.
2. Use weighted sampling on train only.
3. Use augmentation on train only.
4. Train for a fixed **12 epochs**.
5. Do **not** use early stopping/patience.
6. Save the best checkpoint using validation macro-F1.
7. Evaluate the best checkpoint on the test set.

Important: this notebook now requires GPU. If CUDA is not available, it will stop before training so we do not accidentally train on CPU.

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/backup/Intern/combine_skindiseaseclassifier_devraj')
EXPECTED_PYTHON = '/backup/Intern/combine_skindiseaseclassifier_devraj/.venv/bin/python'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

if sys.executable != EXPECTED_PYTHON:
    raise RuntimeError(
        'Wrong notebook kernel selected. Expected: ' + EXPECTED_PYTHON + '\n'
        'Current: ' + sys.executable + '\n'
        'Please switch kernel to: Combine Skin GPU (.venv)'
    )

Project root added: /backup/Intern/combine_skindiseaseclassifier_devraj
Python: /backup/Intern/Devraj/skin_disease_classifier/.venv/bin/python


In [4]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import time

import pandas as pd
import torch
from torch import nn
from torch.amp import GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.models.psa_eca_resnet import build_psa_eca_resnet, count_parameters
from src.training.data import build_dataloaders, class_weight_table
from src.training.engine import train_one_epoch, evaluate
from src.training.checkpoint import save_checkpoint, load_checkpoint, save_class_mapping
from src.training.metrics import (
    save_metrics_json,
    save_classification_report,
    save_confusion_matrix,
    save_predictions_csv,
)
from src.utils.seed import set_seed

PROJECT_ROOT = Path('/backup/Intern/combine_skindiseaseclassifier_devraj')
DATA_DIR = PROJECT_ROOT / 'data' / 'splits'
OUTPUT_ROOT = PROJECT_ROOT / 'training_outputs' / 'psa_eca_resnet'
REPORT_MD_DIR = PROJECT_ROOT / 'markdown_reports'

set_seed(42)
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

Project: /backup/Intern/combine_skindiseaseclassifier_devraj
Data: /backup/Intern/combine_skindiseaseclassifier_devraj/data/splits
Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


## Configuration

This run is fixed to **12 epochs** with **no early stopping**.

Best model is still saved using validation macro-F1, but training continues until epoch 12.

In [5]:
REQUIRE_GPU = True

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA/GPU is not available right now. Do not train on CPU. '
        'First fix GPU access, then rerun this notebook from the top.'
    )

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0  # keep 0 because this server had /dev/shm DataLoader errors
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 12
USE_WEIGHTED_SAMPLER = True
RUN_TAG = 'fixed_12ep'

# Resume option:
# - Keep None for a fresh new run.
# - Set to an existing run folder to continue from last_model.pth.
RESUME_RUN_DIR = OUTPUT_ROOT / 'psa_eca_resnet_fixed_12ep_20260630_042833'
# To start a fresh new run later, change this back to None.

if RESUME_RUN_DIR is not None:
    RUN_DIR = Path(RESUME_RUN_DIR)
    RUN_NAME = RUN_DIR.name
    config = json.loads((RUN_DIR / 'config.json').read_text(encoding='utf-8'))
    config['resumed'] = True
    config['resume_from'] = str(RUN_DIR / 'last_model.pth')
    (RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
else:
    RUN_NAME = f'psa_eca_resnet_{RUN_TAG}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    RUN_DIR = OUTPUT_ROOT / RUN_NAME
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    config = {
        'model': 'psa_eca_resnet',
        'run_name': RUN_NAME,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping': False,
        'patience': None,
        'scheduler': 'CosineAnnealingLR',
        'use_weighted_sampler': USE_WEIGHTED_SAMPLER,
        'best_metric': 'val_f1_macro',
        'resumed': False,
    }
    (RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

print('Run folder:', RUN_DIR)
config

Run folder: /backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/psa_eca_resnet/psa_eca_resnet_fixed_12ep_20260630_042833


{'model': 'psa_eca_resnet',
 'run_name': 'psa_eca_resnet_fixed_12ep_20260630_042833',
 'image_size': 224,
 'batch_size': 32,
 'num_workers': 0,
 'lr': 0.0001,
 'weight_decay': 0.0001,
 'epochs': 12,
 'early_stopping': False,
 'patience': None,
 'scheduler': 'CosineAnnealingLR',
 'use_weighted_sampler': True,
 'best_metric': 'val_f1_macro',
 'resumed': True,
 'resume_from': '/backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/psa_eca_resnet/psa_eca_resnet_fixed_12ep_20260630_042833/last_model.pth'}

## Build DataLoaders

Train uses weighted sampling and augmentation. Validation/test use deterministic transforms only.

In [6]:
bundle = build_dataloaders(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
)

num_classes = len(bundle.class_to_idx)
metric_labels = list(range(num_classes))
class_names = [bundle.idx_to_class[idx] for idx in metric_labels]

print('Classes:', num_classes)
print('Train:', len(bundle.train_dataset))
print('Val:', len(bundle.val_dataset))
print('Test:', len(bundle.test_dataset))

weights_df = class_weight_table(bundle.idx_to_class, bundle.class_counts, bundle.class_weights)
weights_df.to_csv(RUN_DIR / 'class_weights_used.csv', index=False)
save_class_mapping(bundle.class_to_idx, RUN_DIR / 'class_to_idx.json')
weights_df

Classes: 14
Train: 25576
Val: 5478
Test: 5474


,class_idx,class_name,train_count,class_weight
0,0,acne_vulgaris,1727,1.057821
1,1,atopic_dermatitis,1376,1.327658
2,2,basal_cell_carcinoma,4247,0.430152
3,3,contact_dermatitis,1062,1.720205
4,4,drug_eruptions,1115,1.638437
5,5,folliculitis,825,2.214372
6,6,fungal_nail_infections,844,2.164523
7,7,lupus_related_skin_lesions,900,2.029841
8,8,melanoma,6367,0.286926
9,9,plaque_psoriasis,2342,0.780041


In [7]:
images, labels = next(iter(bundle.train_loader))
print('Batch image shape:', images.shape)
print('Batch label shape:', labels.shape)
print('Batch label names:')
pd.Series([bundle.idx_to_class[int(label)] for label in labels]).value_counts()

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
Batch label names:


atopic_dermatitis             6
acne_vulgaris                 4
contact_dermatitis            4
warts                         3
fungal_nail_infections        2
plaque_psoriasis              2
basal_cell_carcinoma          2
melanoma                      2
folliculitis                  2
drug_eruptions                2
lupus_related_skin_lesions    1
vitiligo                      1
tinea_corporis                1
Name: count, dtype: int64

## Build Model

The output should be `[batch_size, 14]`, one score per disease class.

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_psa_eca_resnet(num_classes=num_classes, dropout=0.2).to(device)
total_params, trainable_params = count_parameters(model)

with torch.no_grad():
    test_logits = model(images[:2].to(device))

print('Device:', device)
print('Output shape:', test_logits.shape)
print('Total params:', f'{total_params:,}')
print('Trainable params:', f'{trainable_params:,}')

(RUN_DIR / 'model_summary.txt').write_text(
    f'Model: PSA + ECA ResNet\nTotal params: {total_params:,}\nTrainable params: {trainable_params:,}\nOutput shape: {tuple(test_logits.shape)}\n',
    encoding='utf-8'
)

Device: cuda
Output shape: torch.Size([2, 14])
Total params: 22,358,130
Trainable params: 22,358,130


100

## Train For Fixed 12 Epochs

No early stopping is used. Training runs until epoch 12 unless you manually interrupt it.

If `RESUME_RUN_DIR` is set, this cell loads `last_model.pth`, continues from the next epoch, and appends to the existing `training_log.csv`.

Best checkpoint is still saved by validation macro-F1, not accuracy.

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler('cuda') if device.type == 'cuda' else None

log_path = RUN_DIR / 'training_log.csv'
start_epoch = 1
best_val_f1 = -1.0
best_epoch = None
log_rows = []

if RESUME_RUN_DIR is not None:
    last_checkpoint_path = RUN_DIR / 'last_model.pth'
    if not last_checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint for resume: {last_checkpoint_path}')

    checkpoint = torch.load(last_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if checkpoint.get('optimizer_state_dict') is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    completed_epoch = int(checkpoint['epoch'])
    start_epoch = completed_epoch + 1

    if log_path.exists():
        old_log = pd.read_csv(log_path)
        log_rows = old_log.to_dict('records')
        if len(old_log) > 0:
            best_idx = old_log['val_f1_macro'].idxmax()
            best_val_f1 = float(old_log.loc[best_idx, 'val_f1_macro'])
            best_epoch = int(old_log.loc[best_idx, 'epoch'])

    print(f'Resuming from: {last_checkpoint_path}')
    print(f'Completed epoch: {completed_epoch}')
    print(f'Continuing from epoch: {start_epoch}')
    print(f'Current best val macro-F1: {best_val_f1:.4f} at epoch {best_epoch}')
else:
    print('Starting fresh training run.')

if start_epoch > EPOCHS:
    print(f'This run already completed {EPOCHS} epochs. No more training needed.')
else:
    start_time = time.time()

    for epoch in range(start_epoch, EPOCHS + 1):
        print(f'\nEpoch {epoch}/{EPOCHS}')
        train_metrics = train_one_epoch(
            model=model,
            loader=bundle.train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            metric_labels=metric_labels,
        )
        val_metrics = evaluate(
            model=model,
            loader=bundle.val_loader,
            criterion=criterion,
            device=device,
            return_predictions=False,
            metric_labels=metric_labels,
        )

        val_f1 = float(val_metrics['f1_macro'])
        improved = val_f1 > best_val_f1
        scheduler.step()

        row = {
            'epoch': epoch,
            'lr': optimizer.param_groups[0]['lr'],
            **{f'train_{k}': v for k, v in train_metrics.items()},
            **{f'val_{k}': v for k, v in val_metrics.items()},
            'improved': improved,
        }
        log_rows.append(row)
        pd.DataFrame(log_rows).to_csv(log_path, index=False)

        print(
            f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['f1_macro']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['f1_macro']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        save_checkpoint(
            RUN_DIR / 'last_model.pth',
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            metrics={'train': train_metrics, 'val': val_metrics},
            class_to_idx=bundle.class_to_idx,
            extra=config,
        )

        if improved:
            best_val_f1 = val_f1
            best_epoch = epoch
            save_checkpoint(
                RUN_DIR / 'best_model.pth',
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                metrics={'train': train_metrics, 'val': val_metrics},
                class_to_idx=bundle.class_to_idx,
                extra=config,
            )
            print(f'Saved new best checkpoint at epoch {epoch} with val macro-F1={best_val_f1:.4f}')

    elapsed_minutes = (time.time() - start_time) / 60
    print(f'Finished/resumed fixed 12-epoch training in {elapsed_minutes:.2f} minutes')
    print(f'Best epoch: {best_epoch} with val macro-F1={best_val_f1:.4f}')
    print('Run folder:', RUN_DIR)

Resuming from: /backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/psa_eca_resnet/psa_eca_resnet_fixed_12ep_20260630_042833/last_model.pth
Completed epoch: 2
Continuing from epoch: 3
Current best val macro-F1: 0.4800 at epoch 2

Epoch 3/12


train:   0%|          | 0/800 [00:00<?, ?it/s]

eval:   0%|          | 0/172 [00:00<?, ?it/s]

train_loss=1.3667 train_f1=0.5345 | val_loss=1.3080 val_f1=0.5020 val_acc=0.5646 lr=8.54e-05
Saved new best checkpoint at epoch 3 with val macro-F1=0.5020

Epoch 4/12


train:   0%|          | 0/800 [00:00<?, ?it/s]

...

eval:   0%|          | 0/172 [00:00<?, ?it/s]

train_loss=0.5423 train_f1=0.8237 | val_loss=1.0454 val_f1=0.6253 val_acc=0.6732 lr=1.70e-06
Saved new best checkpoint at epoch 11 with val macro-F1=0.6253

Epoch 12/12


train:   0%|          | 0/800 [00:00<?, ?it/s]

eval:   0%|          | 0/172 [00:00<?, ?it/s]

train_loss=0.5257 train_f1=0.8308 | val_loss=1.0223 val_f1=0.6329 val_acc=0.6773 lr=0.00e+00
Saved new best checkpoint at epoch 12 with val macro-F1=0.6329
Finished/resumed fixed 12-epoch training in 167.54 minutes
Best epoch: 12 with val macro-F1=0.6329
Run folder: /backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/psa_eca_resnet/psa_eca_resnet_fixed_12ep_20260630_042833


In [12]:
import importlib
import src.training.metrics
importlib.reload(src.training.metrics)

from src.training.metrics import (
    save_metrics_json,
    save_classification_report,
    save_confusion_matrix,
    save_predictions_csv,
)

## Evaluate Best Checkpoint On Test Set

The test set is used only after training. This gives the final honest result for this run.

In [13]:
best_checkpoint = RUN_DIR / 'best_model.pth'
checkpoint = load_checkpoint(best_checkpoint, model, map_location=device)
model.to(device)
model.eval()

best_epoch = int(checkpoint['epoch'])
test_metrics = evaluate(
    model=model,
    loader=bundle.test_loader,
    criterion=criterion,
    device=device,
    return_predictions=True,
    metric_labels=metric_labels,
)

y_true = test_metrics.pop('y_true')
y_pred = test_metrics.pop('y_pred')
y_prob = test_metrics.pop('y_prob')

result_payload = {
    'model': 'psa_eca_resnet',
    'run_name': RUN_NAME,
    'best_epoch': best_epoch,
    'best_checkpoint': str(best_checkpoint),
    'test_metrics': test_metrics,
    'config': config,
}
save_metrics_json(result_payload, RUN_DIR / 'metrics.json')
save_classification_report(y_true, y_pred, class_names, RUN_DIR / 'classification_report.txt')
save_confusion_matrix(y_true, y_pred, class_names, RUN_DIR / 'confusion_matrix.png', normalize=False)
save_confusion_matrix(y_true, y_pred, class_names, RUN_DIR / 'confusion_matrix_normalized.png', normalize=True)

image_paths = [sample[0] for sample in bundle.test_dataset.samples]
save_predictions_csv(image_paths, y_true, y_pred, y_prob, bundle.idx_to_class, RUN_DIR / 'predictions_test.csv')

print('Best epoch:', best_epoch)
print(json.dumps(test_metrics, indent=2))
print('Saved test outputs to:', RUN_DIR)

eval:   0%|          | 0/172 [00:00<?, ?it/s]

Best epoch: 12
{
  "accuracy": 0.6799415418341249,
  "precision_macro": 0.6329074588323843,
  "recall_macro": 0.6454232231678888,
  "f1_macro": 0.6373312102840799,
  "f1_weighted": 0.6840228123094767,
  "loss": 1.0225720524551074
}
Saved test outputs to: /backup/Intern/combine_skindiseaseclassifier_devraj/training_outputs/psa_eca_resnet/psa_eca_resnet_fixed_12ep_20260630_042833


## Write Short Markdown Summary

In [14]:
summary_path = REPORT_MD_DIR / f'psa_eca_resnet_{RUN_TAG}_summary.md'
log_df = pd.read_csv(RUN_DIR / 'training_log.csv')
best_row = log_df.loc[log_df['val_f1_macro'].idxmax()]

with summary_path.open('w', encoding='utf-8') as f:
    f.write(f'# PSA + ECA ResNet Fixed 12-Epoch Summary\n\n')
    f.write(f'Run folder: `{RUN_DIR}`\n\n')
    f.write(f'Best epoch by validation macro-F1: **{int(best_row["epoch"])}**\n\n')
    f.write('## Validation At Best Epoch\n\n')
    f.write(f'- Validation accuracy: **{best_row["val_accuracy"]:.4f}**\n')
    f.write(f'- Validation macro-F1: **{best_row["val_f1_macro"]:.4f}**\n')
    f.write(f'- Validation loss: **{best_row["val_loss"]:.4f}**\n\n')
    f.write('## Test Metrics\n\n')
    for key, value in test_metrics.items():
        f.write(f'- {key}: **{float(value):.4f}**\n')
    f.write('\n## Key Files\n\n')
    f.write(f'- Best checkpoint: `{RUN_DIR / "best_model.pth"}`\n')
    f.write(f'- Training log: `{RUN_DIR / "training_log.csv"}`\n')
    f.write(f'- Confusion matrix: `{RUN_DIR / "confusion_matrix.png"}`\n')
    f.write(f'- Classification report: `{RUN_DIR / "classification_report.txt"}`\n')

summary_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/markdown_reports/psa_eca_resnet_fixed_12ep_summary.md')

## Update Project Structure

In [15]:
subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'update_project_structure.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

Updated: /backup/Intern/combine_skindiseaseclassifier_devraj/PROJECT_STRUCTURE.txt


CompletedProcess(args=['python3', '/backup/Intern/combine_skindiseaseclassifier_devraj/scripts/update_project_structure.py'], returncode=0)